# Phần 2. Khám phá dữ liệu Olist

Notebook này khảo sát **dữ liệu nguồn trước xử lý** để mô tả cấu trúc, phát hiện thiếu/trùng, kiểm tra miền giá trị và quan hệ giữa bảng, đồng thời đánh giá các bảng có phù hợp với bài toán dự đoán freight_value ở cấp dòng sản phẩm hay không. Phần này chỉ phân tích và ghi nhận; **không xóa dòng, điền giá trị thiếu, ghép bảng hay tạo dữ liệu đã làm sạch**.

**Trình tự khám phá dữ liệu**

1. **Nguồn và cấu trúc dữ liệu** (2.1): xác định nguồn, phạm vi và cấu trúc các bảng.
2. **Kiểu dữ liệu** (2.2): kiểm tra kiểu dữ liệu và khả năng sử dụng của các trường.
3. **Dữ liệu thiếu và trùng lặp** (2.3): đánh giá mức độ thiếu và tính duy nhất.
4. **Miền giá trị** (2.4): kiểm tra các giá trị có nằm trong miền hợp lệ hay không.
5. **Khóa dữ liệu** (2.5): đánh giá tính duy nhất của khóa và khả năng định danh bản ghi.
6. **Quan hệ giữa các bảng** (2.6): kiểm tra khóa ngoại và tính nhất quán khi liên kết.
7. **Độ phủ thời gian** (2.7): khảo sát giai đoạn có dữ liệu và độ đều giữa các tháng.
8. **Giá trị cực đoan** (2.8): khảo sát các giá trị vượt rào IQR.
9. **Phạm vi sử dụng dữ liệu** (2.9): xác định bảng phù hợp với bài toán và nhận diện nguy cơ rò rỉ dữ liệu.
10. **Tổng hợp phát hiện** (2.10): hệ thống hóa các vấn đề cần lưu ý trước khi chuyển sang bước xử lý dữ liệu.

### 2.1 Đánh giá nguồn và cấu trúc dữ liệu

**Mục đích:** xác định nguồn, giấy phép và cấu trúc dữ liệu gốc trước khi thực hiện các phép kiểm tra chất lượng.

- **Nguồn:** Brazilian E-Commerce Public Dataset by Olist, được công bố trên Kaggle [1]. Bộ dữ liệu gồm các giao dịch thương mại điện tử tại Brazil trong giai đoạn 2016–2018.
- **Giấy phép:** nguồn công bố giấy phép CC BY-NC-SA 4.0 [2]. Khi sử dụng hoặc phân phối sản phẩm phái sinh, cần tuân thủ điều kiện ghi công, phi thương mại và chia sẻ cùng giấy phép.
- **Tính cập nhật:** trang dữ liệu ghi tần suất cập nhật là “Never”. Điều này mô tả trạng thái cập nhật của bản công bố, không thay thế việc kiểm tra đúng phiên bản các tệp CSV đang được sử dụng.
- **Ẩn danh:** theo mô tả nguồn, các mã định danh cá nhân/đối tác trong bộ dữ liệu đã được thay thế bằng mã định danh.

Các kích thước thực tế của chín bảng sẽ được in từ các tệp CSV ở ô code kế tiếp; số liệu trong phần kết luận cần được đối chiếu với kết quả chạy đó.

**Khung đánh giá:** Dự án dùng sáu chiều chất lượng dữ liệu theo DAMA UK [5] làm cơ sở tổ chức các phép kiểm tra; đây là khung tham chiếu, không phải bộ tiêu chuẩn bắt buộc.

| Chiều chất lượng | Cách đánh giá trong dự án | Mục |
|---|---|---|
| Uniqueness (tính duy nhất) | Kiểm tra khóa có trùng lặp hay không | 2.3, 2.5 |
| Completeness (tính đầy đủ) | Thống kê giá trị thiếu | 2.3 |
| Validity (tính hợp lệ) | Kiểm tra kiểu dữ liệu và miền giá trị | 2.2, 2.4 |
| Consistency (tính nhất quán) | Kiểm tra khóa ngoại và quan hệ giữa các bảng | 2.6, 2.9 |
| Timeliness (tính kịp thời) | Khảo sát độ phủ thời gian của dữ liệu | 2.7 |
| Accuracy (tính chính xác) | Chưa thể xác minh trực tiếp do thiếu dữ liệu tham chiếu độc lập | — |

Các phép kiểm tra giúp nhận diện vấn đề tiềm ẩn trong dữ liệu. Riêng các giá trị vượt rào IQR chỉ được xem là giá trị cực đoan cần xem xét, không mặc nhiên được kết luận là không hợp lệ hoặc không chính xác.

**Cấu trúc các bảng:** nguồn gồm chín tệp CSV. Trong notebook, mỗi tệp được đọc thành một bảng và gọi bằng tên ngắn ở cột đầu.

| Bảng | Tệp CSV | Nội dung | Vai trò được xem xét |
|---|---|---|---|
| items | olist_order_items_dataset | Từng sản phẩm trong đơn, gồm price và freight_value | Bảng quan sát và biến mục tiêu |
| orders | olist_orders_dataset | Đơn hàng, trạng thái và các mốc thời gian | Bổ sung trạng thái/thời điểm đặt hàng |
| products | olist_products_dataset | Danh mục, cân nặng và kích thước sản phẩm | Thông tin sản phẩm |
| customers | olist_customers_dataset | Mã bưu chính và bang của khách | Vị trí khách |
| sellers | olist_sellers_dataset | Mã bưu chính và bang của người bán | Vị trí người bán |
| geo | olist_geolocation_dataset | Tọa độ theo mã bưu chính | Xem xét thông tin vị trí |
| cat_map | product_category_name_translation | Bản dịch danh mục sang tiếng Anh | Tên danh mục |
| payments | olist_order_payments_dataset | Thanh toán của đơn | Kiểm tra nguy cơ rò rỉ ở mục 2.9 |
| reviews | olist_order_reviews_dataset | Đánh giá của khách sau khi nhận hàng | Đánh giá tính phù hợp theo thời điểm dự đoán |

![Sơ đồ quan hệ giữa các bảng](https://lh3.googleusercontent.com/d/1AtOtSDQlSSoF2KCiw7zWY5kM3-vH7wFj)

Hình 2.1. Sơ đồ quan hệ giữa tám bảng theo trang dữ liệu Olist; sơ đồ không vẽ bảng cat_map (tệp product_category_name_translation).

Nạp thư viện, mount Google Drive, đọc chín tệp CSV và in kích thước từng bảng. Nếu đường dẫn lưu CSV khác, cập nhật biến raw trước khi chạy.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

from google.colab import drive
drive.mount('/content/drive/')

raw = '/content/drive/MyDrive/Colab Notebooks/'

orders    = pd.read_csv(raw + 'olist_orders_dataset.csv')
items     = pd.read_csv(raw + 'olist_order_items_dataset.csv')
products  = pd.read_csv(raw + 'olist_products_dataset.csv')
customers = pd.read_csv(raw + 'olist_customers_dataset.csv')
geo       = pd.read_csv(raw + 'olist_geolocation_dataset.csv')
sellers   = pd.read_csv(raw + 'olist_sellers_dataset.csv')
cat_map   = pd.read_csv(raw + 'product_category_name_translation.csv')
payments  = pd.read_csv(raw + 'olist_order_payments_dataset.csv')
reviews   = pd.read_csv(raw + 'olist_order_reviews_dataset.csv')

tables = {
    'orders': orders, 'items': items, 'products': products, 'customers': customers,
    'geo': geo, 'sellers': sellers, 'cat_map': cat_map,
}

for name, t in {**tables, 'payments': payments, 'reviews': reviews}.items():
    print(f'{name:<10}: {t.shape}')

Mounted at /content/drive/
orders    : (99441, 8)
items     : (112650, 7)
products  : (32951, 9)
customers : (99441, 5)
geo       : (1000163, 5)
sellers   : (3095, 4)
cat_map   : (71, 2)
payments  : (103886, 5)
reviews   : (99224, 7)


**Kết luận:** 9 tệp được đọc đầy đủ; kích thước từng bảng như sau.

| Bảng | Số dòng | Số cột |
|---|---|---|
| orders | 99,441 | 8 |
| items | 112,650 | 7 |
| products | 32,951 | 9 |
| customers | 99,441 | 5 |
| geo | 1,000,163 | 5 |
| sellers | 3,095 | 4 |
| cat_map | 71 | 2 |
| payments | 103,886 | 5 |
| reviews | 99,224 | 7 |

### 2.2 Kiểm tra kiểu dữ liệu

**Câu hỏi:** Các cột có kiểu dữ liệu phù hợp với ý nghĩa và cách sử dụng của chúng không?

In [2]:
for name,t in {**tables,'payments':payments,'reviews':reviews}.items():
  print('---',name,'---')
  print(t.dtypes)

--- orders ---
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
--- items ---
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object
--- products ---
product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object
--- customers ---
customer_id    

**Kết luận:** 8 cột ngày ở kiểu chữ, chưa khớp ý nghĩa; các cột còn lại có kiểu phù hợp.

| Bảng | Cột | Kiểu hiện tại |
|---|---|---|
| orders | order_purchase_timestamp | object |
| orders | order_approved_at | object |
| orders | order_delivered_carrier_date | object |
| orders | order_delivered_customer_date | object |
| orders | order_estimated_delivery_date | object |
| items | shipping_limit_date | object |
| reviews | review_creation_date | object |
| reviews | review_answer_timestamp | object |

### 2.3 Kiểm tra dữ liệu thiếu và trùng lặp

**Câu hỏi:** bảng nào, cột nào có giá trị thiếu và thiếu bao nhiêu; bảng nào có dòng trùng lặp?

Đếm ô thiếu theo cột của 7 bảng.

In [3]:
for name, t in tables.items():
    miss = t.isnull().sum()
    print('---', name, '---')
    if miss.sum() > 0:
        print(miss[miss > 0])
    else:
        print('khong thieu')

--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
--- items ---
khong thieu
--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
--- customers ---
khong thieu
--- geo ---
khong thieu
--- sellers ---
khong thieu
--- cat_map ---
khong thieu


**Kết luận:** 2 trong 7 bảng có giá trị thiếu.

| Bảng | Cột | Số dòng thiếu | Tỷ lệ |
|---|---|---|---|
| orders | order_approved_at | 160 | 0.16% |
| orders | order_delivered_carrier_date | 1,783 | 1.79% |
| orders | order_delivered_customer_date | 2,965 | 2.98% |
| products | product_category_name | 610 | 1.85% |
| products | product_name_lenght | 610 | 1.85% |
| products | product_description_lenght | 610 | 1.85% |
| products | product_photos_qty | 610 | 1.85% |
| products | product_weight_g | 2 | 0.01% |
| products | product_length_cm | 2 | 0.01% |
| products | product_height_cm | 2 | 0.01% |
| products | product_width_cm | 2 | 0.01% |

**Dòng trùng lặp:** bảng nào có dòng trùng hoàn toàn?

Đếm dòng trùng hoàn toàn của từng bảng.

In [4]:
for name, t in tables.items():
    print(f'trung toan dong {name:<10}: {t.duplicated().sum()}')

trung toan dong orders    : 0
trung toan dong items     : 0
trung toan dong products  : 0
trung toan dong customers : 0
trung toan dong geo       : 261831
trung toan dong sellers   : 0
trung toan dong cat_map   : 0


**Kết luận:** Bảng geo có 261,831 dòng trùng hoàn toàn.

### 2.4 Kiểm tra giá trị sai miền

**Câu hỏi:** giá trị quan sát có nằm ngoài miền hợp lệ được xác định trước hay không?

**Miền kiểm tra:**

- price lớn hơn 0 và freight_value không âm.
- Cân nặng và ba kích thước của sản phẩm lớn hơn 0.
- Mã bang thuộc danh sách 27 đơn vị hành chính của Brazil.
- Tọa độ nằm trong khung bao xấp xỉ của Brazil: vĩ độ từ -34 đến 5.3, kinh độ từ -74 đến -28 [3][4]. Đây là kiểm tra sàng lọc bằng khung bao, không xác minh vị trí thực tế của từng điểm.

In [5]:
print('price <= 0         :', (items['price'] <= 0).sum())
print('freight_value < 0  :', (items['freight_value'] < 0).sum())
print('freight_value = 0  :', (items['freight_value'] == 0).sum())
print('weight_g <= 0      :', (products['product_weight_g'] <= 0).sum())
print('length_cm <= 0     :', (products['product_length_cm'] <= 0).sum())
print('height_cm <= 0     :', (products['product_height_cm'] <= 0).sum())
print('width_cm <= 0      :', (products['product_width_cm'] <= 0).sum())

UF = ['AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO']
print('customer_state khong hop le   :', (~customers['customer_state'].isin(UF)).sum())
print('seller_state khong hop le     :', (~sellers['seller_state'].isin(UF)).sum())
print('geolocation_state khong hop le:', (~geo['geolocation_state'].isin(UF)).sum())

trong_khung = geo['geolocation_lat'].between(-34, 5.3) & geo['geolocation_lng'].between(-74, -28)
print('geo ngoai khung bao Brazil    :', (~trong_khung).sum())

price <= 0         : 0
freight_value < 0  : 0
freight_value = 0  : 383
weight_g <= 0      : 4
length_cm <= 0     : 0
height_cm <= 0     : 0
width_cm <= 0      : 0
customer_state khong hop le   : 0
seller_state khong hop le     : 0
geolocation_state khong hop le: 0
geo ngoai khung bao Brazil    : 31


**Kết luận:**
- **Giá trị số:** 4 sản phẩm có product_weight_g bằng 0; 383 dòng có freight_value bằng 0 (không âm nên không vi phạm miền, nhưng cần xem xét).
- **Tọa độ địa lý:** 31 tọa độ nằm ngoài khung bao Brazil.

### 2.5 Kiểm tra khóa

**Câu hỏi:** khóa của mỗi bảng có duy nhất không? Khóa lặp làm một bản ghi tương ứng với nhiều dòng của bảng kia, nên một giá trị có thể bị tính nhiều lần.

Kiểm tra tính duy nhất của khóa từng bảng, và đếm số dòng so với số mã bưu chính khác nhau của geo.

In [11]:
print('orders.order_id                :', orders['order_id'].is_unique)
print('products.product_id            :', products['product_id'].is_unique)
print('cat_map.product_category_name  :', cat_map['product_category_name'].is_unique)
print('customers.customer_id          :', customers['customer_id'].is_unique)
print('sellers.seller_id              :', sellers['seller_id'].is_unique)
print('items(order_id, order_item_id) :', not items[['order_id', 'order_item_id']].duplicated().any())
print('geo.geolocation_zip_code_prefix:', geo['geolocation_zip_code_prefix'].is_unique)

orders.order_id                : True
products.product_id            : True
cat_map.product_category_name  : True
customers.customer_id          : True
sellers.seller_id              : True
items(order_id, order_item_id) : True
geo.geolocation_zip_code_prefix: False


Kiểm tra khóa số lượng trùng khóa geo số mã bưu chính

In [12]:
print('geo so dong                    :', len(geo))
print('geo so ma buu chinh            :', geo['geolocation_zip_code_prefix'].nunique())

geo so dong                    : 1000163
geo so ma buu chinh            : 19015


**Kết luận:** khóa của 6 bảng (kể cả khóa ghép order_id và order_item_id của items) đều duy nhất; riêng geolocation_zip_code_prefix của geo không duy nhất, bảng có 1,000,163 dòng nhưng chỉ 19,015 mã bưu chính khác nhau.

### 2.6 Kiểm tra mối quan hệ giữa các bảng

**Câu hỏi:** mỗi giá trị khóa ngoại có tồn tại ở bảng được tham chiếu không? Khóa ngoại không khớp làm dòng ghép bị thiếu giá trị từ bảng kia.

Đếm giá trị khóa ngoại không có trong bảng được tham chiếu.

In [7]:
cats = products['product_category_name'].dropna()
print('items.product_id   -> products.product_id  :', (~items['product_id'].isin(products['product_id'])).sum())
print('items.order_id     -> orders.order_id      :', (~items['order_id'].isin(orders['order_id'])).sum())
print('items.seller_id    -> sellers.seller_id    :', (~items['seller_id'].isin(sellers['seller_id'])).sum())
print('orders.customer_id -> customers.customer_id:', (~orders['customer_id'].isin(customers['customer_id'])).sum())
print('products.product_category_name -> cat_map  :', (~cats.isin(cat_map['product_category_name'])).sum())
print('customers.zip -> geo                       :', (~customers['customer_zip_code_prefix'].isin(geo['geolocation_zip_code_prefix'])).sum())
print('sellers.zip   -> geo                       :', (~sellers['seller_zip_code_prefix'].isin(geo['geolocation_zip_code_prefix'])).sum())

items.product_id   -> products.product_id  : 0
items.order_id     -> orders.order_id      : 0
items.seller_id    -> sellers.seller_id    : 0
orders.customer_id -> customers.customer_id: 0
products.product_category_name -> cat_map  : 13
customers.zip -> geo                       : 278
sellers.zip   -> geo                       : 7


**Kết luận:** 4 quan hệ theo mã định danh toàn vẹn; cat_map thiếu bản dịch của 13 sản phẩm và geo chưa phủ hết mã bưu chính.

| Quan hệ | Giá trị không có ở bảng cha |
|---|---|
| products - cat_map | 13 sản phẩm |
| customers - geo (mã bưu chính) | 278 dòng của customers |
| sellers - geo (mã bưu chính) | 7 người bán |

### 2.7 Khảo sát độ phủ thời gian

**Câu hỏi:** dữ liệu phủ giai đoạn nào và số đơn có đều giữa các tháng không?

Đếm số đơn theo tháng đặt hàng và in thời điểm đặt hàng sớm nhất, muộn nhất.

In [8]:
t_mua = pd.to_datetime(orders['order_purchase_timestamp'])
print('don som nhat :', t_mua.min())
print('don muon nhat:', t_mua.max())
print(t_mua.dt.to_period('M').value_counts().sort_index())

don som nhat : 2016-09-04 21:15:19
don muon nhat: 2018-10-17 17:30:18
order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64


**Kết luận:** dữ liệu trải từ 04/09/2016 đến 17/10/2018, liên tục từ 2017-01 đến 2018-08, còn đầu và cuối chuỗi chỉ có rất ít đơn.

| Giai đoạn | Số đơn theo tháng |
|---|---|
| 2016-09 đến 2016-12 | 4; 324; không có đơn (2016-11); 1 |
| 2017-01 đến 2018-08 | từ 800 đến 7,544 |
| 2018-09 đến 2018-10 | 16; 4 |

### 2.8 Khảo sát giá trị cực đoan

**Câu hỏi:** những biến số nào có nhiều giá trị nằm ngoài rào trên IQR?

**Cách làm:** rào trên Tukey là Q3 + 1.5 × IQR; đếm giá trị vượt rào của price, freight_value (bảng items), cân nặng và ba kích thước (bảng products), mỗi biến tính trên bảng nguồn của nó.

**Lưu ý:** giá trị vượt rào chỉ là dấu hiệu cực đoan, không phải phép đo Accuracy và không chứng minh giá trị sai. Chỉ xét rào trên vì cả 6 biến chỉ nhận giá trị không âm nên rào dưới của 5 biến là số âm; riêng freight_value có rào dưới 0.975 và 521 giá trị thấp hơn, trong đó có 383 dòng bằng 0 đã nêu ở 2.4.

Tính rào trên và đếm số giá trị vượt rào của từng biến.

In [9]:
cot = pd.concat([items[['price', 'freight_value']],
                 products[['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']]], axis=1)
q1, q3 = cot.quantile(0.25), cot.quantile(0.75)
rao_tren = q3 + 1.5 * (q3 - q1)
vuot = (cot > rao_tren).sum()
print(pd.DataFrame({'rao_tren': rao_tren, 'so_vuot': vuot, 'ty_le_%': (vuot / cot.count() * 100).round(2)}))

                   rao_tren  so_vuot  ty_le_%
price               277.400     8427     7.48
freight_value        33.255    11613    10.31
product_weight_g   4300.000     4551    13.81
product_length_cm    68.000     1380     4.19
product_height_cm    40.500     1892     5.74
product_width_cm     52.500      912     2.77


**Kết luận:** cả 6 biến đều có giá trị vượt rào trên, tỷ lệ từ 2.77% đến 13.81%.

| Biến | Rào trên IQR | Số giá trị vượt rào | Tỷ lệ |
|---|---:|---:|---:|
| price | 277.400 | 8,427 | 7.48% |
| freight_value | 33.255 | 11,613 | 10.31% |
| product_weight_g | 4,300.000 | 4,551 | 13.81% |
| product_length_cm | 68.000 | 1,380 | 4.19% |
| product_height_cm | 40.500 | 1,892 | 5.74% |
| product_width_cm | 52.500 | 912 | 2.77% |

### 2.9 Kiểm tra bảng payments và phạm vi sử dụng dữ liệu

**Câu hỏi:** payment_value của bảng payments có dùng làm đặc trưng được không?

**Giả thuyết:** với mỗi đơn có ở cả items và payments, so tổng payment_value với tổng price cộng freight_value.

In [10]:
tong_hang = (items['price'] + items['freight_value']).groupby(items['order_id']).sum().round(2)
tong_tt = payments.groupby('order_id')['payment_value'].sum().round(2)
chung = tong_hang.index.intersection(tong_tt.index)
print('so don co ca items va payments:', len(chung))
print('ty le khop:', round((tong_hang[chung] == tong_tt[chung]).mean() * 100, 2), '%')

so don co ca items va payments: 98665
ty le khop: 99.42 %


**Kết luận:** 99.42% đơn có payment_value đúng bằng price cộng freight_value, nên payment_value chứa phí vận chuyển và không được dùng làm đặc trưng.

**Đánh giá khả năng sử dụng bảng:**

| Bảng | Đánh giá | Căn cứ |
|---|---|---|
| items | Dùng làm bảng quan sát | Chứa freight_value và khóa ghép (order_id, order_item_id) |
| orders | Có thể dùng một phần trường | Trạng thái và thời điểm đặt hàng; các mốc giao hàng là thông tin sau thời điểm đặt hàng |
| products | Có thể dùng một phần trường | Cân nặng, kích thước và danh mục sản phẩm; cần chú ý giá trị thiếu/sai miền ở mục 2.3–2.4 |
| customers | Có thể dùng một phần trường | Mã bưu chính và bang; mã định danh khách hàng và thành phố không được chọn trong phạm vi xem xét này |
| sellers | Có thể dùng một phần trường | Mã bưu chính và bang; thành phố không được chọn trong phạm vi xem xét này |
| geo | Có thể dùng có điều kiện | Có tọa độ, nhưng mã bưu chính không duy nhất, có điểm ngoài khung bao và có mã không được phủ |
| cat_map | Có thể dùng có điều kiện | Có bản dịch danh mục, nhưng một số danh mục không có bản dịch |
| payments | Không chọn làm đặc trưng | payment_value có quan hệ gần như trực tiếp với price + freight_value, tạo nguy cơ rò rỉ (2.9) |
| reviews | Không chọn làm đặc trưng | Đánh giá được tạo sau khi khách nhận hàng, không sẵn có ở thời điểm dự đoán đã giả định |

Các trường cần loại khỏi tập đặc trưng theo giả định thời điểm dự đoán, dù chúng vẫn có giá trị cho phân tích hậu kiểm:

- orders: order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date.
- products: product_name_lenght, product_description_lenght, product_photos_qty.
- customers: customer_unique_id, customer_city.
- sellers: seller_city.
- geo: geolocation_city.

Đây là đánh giá phạm vi sử dụng, không tạo ra bảng dữ liệu mới và không thực hiện làm sạch.

### 2.10 Tổng hợp vấn đề phát hiện

Bảng dưới tổng hợp những điểm được ghi nhận từ mục 2.2 đến 2.9. Số liệu tĩnh trong bảng cần được đối chiếu với kết quả sau khi chạy toàn bộ notebook trên đúng chín tệp CSV. Bảng này chỉ là **danh mục phát hiện EDA**, không phải nhật ký các thao tác làm sạch.

| STT | Mục | Loại phát hiện | Bảng/cột | Số liệu ghi nhận | Ý nghĩa/giới hạn |
|---:|---|---|---|---|---|
| 1 | 2.2 | Kiểu dữ liệu | orders (5 cột ngày), items.shipping_limit_date, reviews (2 cột ngày) | 8 cột ngày ở kiểu chữ | Cần chuyển kiểu nếu một quy trình sau này cần tính toán thời gian; notebook này không chuyển kiểu |
| 2 | 2.3 | Thiếu dữ liệu | orders.order_approved_at | 160 dòng thiếu | Độ đầy đủ của mốc duyệt đơn bị hạn chế |
| 3 | 2.3 | Thiếu dữ liệu | orders.order_delivered_carrier_date | 1,783 dòng thiếu | Độ đầy đủ của mốc giao cho vận chuyển bị hạn chế |
| 4 | 2.3 | Thiếu dữ liệu | orders.order_delivered_customer_date | 2,965 dòng thiếu | Độ đầy đủ của mốc giao khách bị hạn chế |
| 5 | 2.3 | Thiếu dữ liệu | products: danh mục và ba trường mô tả | 610 ô thiếu ở mỗi trường | Là số ô thiếu theo từng trường, chưa tính riêng số sản phẩm bị ảnh hưởng |
| 6 | 2.3 | Thiếu dữ liệu | products: cân nặng và ba kích thước | 2 ô thiếu ở mỗi trường | Thiếu thông tin vật lý có thể ảnh hưởng nếu dùng các trường này |
| 7 | 2.3 | Dòng trùng hoàn toàn | geo | 261,831 dòng theo thống kê ghi nhận | Trùng hoàn toàn không giống vấn đề nhiều tọa độ trên cùng mã bưu chính |
| 8 | 2.4 | Giá trị cần điều tra, không tự động sai miền | items.freight_value | 383 dòng bằng 0 theo thống kê ghi nhận | Không âm nên không vi phạm điều kiện miền đã đặt ra; nguyên nhân chưa xác định |
| 9 | 2.4 | Sai miền theo quy tắc đã đặt | products.product_weight_g | 4 sản phẩm có cân nặng bằng 0 theo thống kê ghi nhận | Cần phân biệt với giá trị thiếu; chưa chỉnh sửa tại phần EDA |
| 10 | 2.4 | Điểm cần rà soát | geo tọa độ | 31 điểm ngoài khung bao Brazil theo thống kê ghi nhận | Khung bao là kiểm tra sàng lọc, không đủ để xác nhận chắc chắn dữ liệu sai |
| 11 | 2.5 | Khóa không duy nhất | geo.geolocation_zip_code_prefix | 1,000,163 dòng cho 19,015 mã | Ghép trực tiếp có thể làm nhân bản dòng quan sát |
| 12 | 2.6 | Khóa tham chiếu không khớp | products.product_category_name so với cat_map | 13 sản phẩm theo thống kê ghi nhận, không tính danh mục thiếu | Không có bản dịch cho một số mã danh mục |
| 13 | 2.6 | Mã bưu chính không được phủ | Khách hàng so với geo | 278 dòng theo thống kê ghi nhận | Không tìm thấy mã bưu chính trong bảng tọa độ |
| 14 | 2.6 | Mã bưu chính không được phủ | Người bán so với geo | 7 người bán theo thống kê ghi nhận | Không tìm thấy mã bưu chính trong bảng tọa độ |
| 15 | 2.7 | Độ phủ thời gian không đồng đều | orders.order_purchase_timestamp | 2016 chỉ có vài tháng với lượng đơn thấp; cuối 2018 cũng ít đơn | Mô tả độ phủ, không đủ kết luận timeliness hoặc nguyên nhân |
| 16 | 2.8 | Giá trị vượt rào IQR | items.price, items.freight_value, products cân nặng/kích thước | 7.48%; 10.31%; 13.81%; 4.19%; 5.74%; 2.77% theo thống kê ghi nhận | Là dấu hiệu phân phối cần xem xét, không phải tỷ lệ dữ liệu sai hay phép đo Accuracy |
| 17 | 2.9 | Nguy cơ rò rỉ dữ liệu | payments.payment_value | 99.42% trong 98,665 đơn khớp tổng price + freight_value theo thống kê ghi nhận | Bằng chứng thực nghiệm về nguy cơ; không khẳng định quan hệ đúng tuyệt đối cho mọi đơn |

**Giới hạn của EDA:** chưa có nguồn tham chiếu độc lập để xác nhận tính chính xác thực tế; một số phép kiểm tra địa lý dựa trên khung bao; phép đếm khóa ngoại không thay thế kiểm tra chất lượng từng giá trị; và những số liệu ghi nhận cần được xác nhận bằng cách chạy lại notebook trên cùng phiên bản CSV. Không có dòng nào bị xóa, điền hoặc biến đổi trong phần này.

### Tài liệu tham khảo

[1] Olist, Brazilian E-Commerce Public Dataset by Olist, Kaggle. https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data (truy cập ngày 28/10/2026)

[2] Creative Commons, Attribution-NonCommercial-ShareAlike 4.0 International. https://creativecommons.org/licenses/by-nc-sa/4.0/

[3] IBGE, Pontos extremos do Brasil. https://educa.ibge.gov.br/criancas/voce-sabia/23101-pontos-extremos.html

[4] Wikipedia, List of extreme points of Brazil. https://en.wikipedia.org/wiki/List_of_extreme_points_of_Brazil

[5] UK Government Data Quality Hub, Meet the data quality dimensions (sáu chiều theo DAMA UK). https://www.gov.uk/government/news/meet-the-data-quality-dimensions